### **IMPORTS NECESARIOS**

In [1]:
# Conduct the imports
import torch
import numpy as np 
import torch.nn as nn
import matplotlib.pyplot as plt

### **MOE BLOCK CREATION**

#### **ROUTER IMPLEMENTATION**

In [2]:
class Router(nn.Module):
    def __init__(self, dk:int, n_experts:int, k:int):


        # Initialize parent
        super().__init__()

        # Define the parameters
        self.n_experts = n_experts

        self.k = min(k, self.n_experts)

        # Define the linear layer (B*N, dk) --> (B*N, nExperts)
        self.linear = nn.Linear(in_features = dk, out_features = n_experts)

        # Define the softmax
        self.softmax = nn.Softmax(dim = -1)


    def forward(self, x:torch.Tensor)->torch.Tensor:

        '''  
            By setting n_experts to one during training, we force
            the router to select the expert we want
        
        '''
        # Get the first dim
        first_dim = x.shape[0]
        # Input (B*N_tokens, dk) --> (B*N_tokens, NExperts)
        x = self.linear(x)
        print(f'(router) Output of the linear:{x.shape}')

        # Get the top-k over logits
        top_k_values, top_k_indices  = torch.topk(x, k = self.k, dim = 1)
        print(f'(router) Output of the top_k:{top_k_values.shape}--({top_k_indices.shape})')
        
        # Create -inf matrix
        inf_mask = torch.full_like(x, fill_value = float('-inf'))
        print(f'(router) Aspect of the -inf mask:{inf_mask.shape}')

        # Unmasking experts belonging to the top-k
        print(f'(router) BUILDING INDEXING MECHANISM')
        # 1. Create arange for indexing (B*N_tokens, 1)
        first_index = torch.arange(0, first_dim).unsqueeze(-1)
        print(f'(router) ·Aspect of the first index:{first_index.shape}')
        # 2. Second index shape (B*Ntokens, experts) -> allowed broadcast
        print(f'(router) ·Aspect of the second index:{top_k_indices.shape}')
        # 3. Set selected experts to their values 
        masked_logits = inf_mask[first_index, top_k_indices] = top_k_values
        print(f'(router) ·Aspect of the later inf maskk:{inf_mask.shape}')

        # Compute the probabilities 
        top_k_weights = self.softmax(masked_logits)
        print(f'(router) Aspect of the normalized weights:{top_k_weights.shape}')


        return top_k_weights, top_k_indices

In [3]:
# Define the params
dk = 10
n_experts = 5
k = 2

# Initialize the router
router = Router(dk = dk, n_experts = n_experts, k = k)

# Call the router 
mock_tensor = torch.randn(10, dk)
top_k_weights, top_k_indices = router(mock_tensor)


(router) Output of the linear:torch.Size([10, 5])
(router) Output of the top_k:torch.Size([10, 2])--(torch.Size([10, 2]))
(router) Aspect of the -inf mask:torch.Size([10, 5])
(router) BUILDING INDEXING MECHANISM
(router) ·Aspect of the first index:torch.Size([10, 1])
(router) ·Aspect of the second index:torch.Size([10, 2])
(router) ·Aspect of the later inf maskk:torch.Size([10, 5])
(router) Aspect of the normalized weights:torch.Size([10, 2])


#### **EXPERT BLOCK IMPLEMENTATION**

***ACTIVATION FUNCTION***

In [4]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

***EXPERT MAIN BLOCK***

In [ ]:
class Expert(nn.Module):

    def __init__(self, dk:int, dh:int,  dout:int):

        # Initialize parent
        super().__init__()

        # Define the FFN
        self.l1 = nn.Linear(in_features = dk, out_features = dh)
        self.l2 = nn.Linear(in_features = dh , out_features =  dout)

        # Define the activation function
        self.activation = GELU()


    def forward(self, x:torch.Tensor)->torch.Tensor:

        # First Linear Layer (B*N, dk)-->(B*N, dhidden)
        x = self.activation(self.l1(x))
        print(f'(expert) Output of the first linear:{x.shape}')


        # Second Linear Layer (B* N, dk)-->(B*N, dhidden)
        x = self.activation(self.l2(x))
        print(f'(expert) Output of the second linear:{x.shape}')

        return x


In [6]:
# Define the params
dk = 10
dh = 100
dout = 10

# Initialize the router
router = Expert(dk = dk, dh = dh,  dout = dout)

# Call the router 
mock_tensor = torch.randn(10, dk)
top_k_weights = router(mock_tensor)


(expert) Output of the first linear:torch.Size([10, 100])
(expert) Output of the second linear:torch.Size([10, 10])


#### **MOE LAYER**

In [7]:
class MOE(nn.Module):

    def __init__(self, args:dict[str, int]):
        # Initialize the main module 
        super().__init__()

        # Get the args
        self.input_dim = args.get('input_dim', 768)
        self.n_exp = args.get('experts', 1)
        self.k_routing = args.get('k_routing', 1)
        self.emb_dim = args.get('emb_dim', 768 )

        
        # Initialize the gating mechanism
        self.gate = Router(dk = self.input_dim, n_experts = self.n_exp, k = self.k_routing)

        # Initialize the experts
        self.experts:list = [Expert(dk = self.input_dim, dh = self.emb_dim, dout = self.input_dim) for _ in range(self.n_exp)]


    def forward(self, x:torch.Tensor)->torch.Tensor:

        # Keep original shape
        n_batches, tokens,_ = x.shape
        #1. Get the top-k experts
        print(f'(MOE) CALLING GATING MECHANISM')
        # Flatten before passing (B, N, din)-->(B*N, di)
        x =  x.view(-1, self.input_dim)
        # Results size (B*N, din)- (B*N, din)
        weights, selected_experts = self.gate(x)

        # 2. Design the results tensor (B*N, dout = din)
        results_tensor = torch.zeros_like(x)
        print(f'(MOE) Aspect of the container for output tensor:', results_tensor.shape)

        # 3. Iterate over the experts
        for i, expert in enumerate(self.experts):
            print(f'(MOE) EXPERT ({i})')

            # Get matching tokens
            token_idx, expert_idx  =  torch.where(selected_experts == i)
            # Check if it is not none -> avoid calling not used experts 
            if token_idx is None:
                continue
            # Get expert output (batch*N, dout)
            expert_output = expert(x[token_idx, :])
            # Get expert weights (N, 1)
            expert_weigths = weights[token_idx, expert_idx].unsqueeze(-1)
            print(f'(expert) Aspect of the expert weights:', expert_weigths.shape)

            # Modify the current results tracking --> (N, 1)*(N, )
            results_tensor[token_idx] += expert_weigths*expert_output

        #4. Reshape the  output (B*N, din = dout) - >(B, N, din = dout)
        final_output = results_tensor.view(n_batches, tokens, self.input_dim)
        print(f'(MOE) Aspect of the  output tensor:', final_output.shape)



In [8]:
# Define the params
params = {"input_dim":768, "experts":2, "k_routing":1, "emb_dim": 768}

# Initialize the MOE
moe = MOE(params)

# Create mock tensor
bs = 10
tokens = 100
mock_tensor = torch.randn(10, 100, 768)
moe(mock_tensor)



(MOE) CALLING GATING MECHANISM
(router) Output of the linear:torch.Size([1000, 2])
(router) Output of the top_k:torch.Size([1000, 1])--(torch.Size([1000, 1]))
(router) Aspect of the -inf mask:torch.Size([1000, 2])
(router) BUILDING INDEXING MECHANISM
(router) ·Aspect of the first index:torch.Size([1000, 1])
(router) ·Aspect of the second index:torch.Size([1000, 1])
(router) ·Aspect of the later inf maskk:torch.Size([1000, 2])
(router) Aspect of the normalized weights:torch.Size([1000, 1])
(MOE) Aspect of the container for output tensor: torch.Size([1000, 768])
(MOE) EXPERT (0)
(expert) Output of the first linear:torch.Size([506, 768])
(expert) Output of the second linear:torch.Size([506, 768])
(expert) Aspect of the expert weights: torch.Size([506, 1])
(MOE) EXPERT (1)
(expert) Output of the first linear:torch.Size([494, 768])
(expert) Output of the second linear:torch.Size([494, 768])
(expert) Aspect of the expert weights: torch.Size([494, 1])
(MOE) Aspect of the  output tensor: torch